# ==============================================================================
# 🚢 MARITIME EVACUATION SURVIVAL: TITANIC VIA GRADIENT BOOSTING CLASSIFIER
# ==============================================================================
### Core Theoretical Principles:
Gradient Tree Boosting optimizes binary classification by minimizing **Bernoulli Deviance** (Log-Loss):
$$L(y, F(x)) = \log(1 + e^{F(x)}) - y \cdot F(x)$$

Each sequential tree $h_m(x)$ is fitted to the **pseudo-residuals** (negative gradients):
$$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F_{m-1}} = y_i - p_{i, m-1}$$

The terminal leaf output values are determined by a second-order Taylor expansion (one-step Newton-Raphson approximation):
$$\gamma_{jm} \approx \frac{\sum_{x_i \in R_{jm}} r_{im}}{\sum_{x_i \in R_{jm}} p_{i, m-1}(1 - p_{i, m-1})}$$

Cumulative predictions update smoothly via shrinkage:
$$F_m(x) = F_{m-1}(x) + \nu \sum_{j=1}^{J_m} \gamma_{jm} \mathbb{I}(x \in R_{jm})$$


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT & REPRODUCIBILITY SEED
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Enterprise Environment initialized.")


✅ STEP 1: Enterprise Environment initialized.


## 📥 STEP 2 & 3: RAW DATASET INGESTION & 3-SECOND HEALTH AUDIT
Loading Titanic passenger manifest (891 passenger records).


In [2]:
# STEP 2 & 3: INGESTION & AUDIT
data_path = 'data/titanic/Titanic-Dataset.csv'
df = pd.read_csv(data_path)

print(f"📊 Dataset Dimensions: {df.shape[0]} passengers, {df.shape[1]} attributes")
print(f"Survival Distribution:\n{df['Survived'].value_counts(normalize=True).round(3)}")
print(f"Missing Values Summary:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
df.head(3)


📊 Dataset Dimensions: 891 passengers, 12 attributes
Survival Distribution:
Survived
0    0.616
1    0.384
Name: proportion, dtype: float64
Missing Values Summary:
Age         177
Cabin       687
Embarked      2
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 🧹 STEP 4 & 5: STRICT SEGREGATION & HYGIENE
Dropping identification noise (`PassengerId`, `Name`, `Ticket`, `Cabin`) and separating features and target.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
y = df['Survived'].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution :\n{y.value_counts()}")


Feature matrix shape: (891, 7)
Target distribution :
Survived
0    549
1    342
Name: count, dtype: int64


## 🔒 STEP 6: ZERO-DATA-LEAKAGE STRATIFIED SPLIT
Stratified 80/20 train/test split.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train Cohort: {X_train.shape[0]} passengers")
print(f"Test Cohort : {X_test.shape[0]} passengers")


Train Cohort: 712 passengers
Test Cohort : 179 passengers


## ⚙️ STEP 7: ROBUST COLUMNTRANSFORMER PREPROCESSOR
Handling missing numerical ages via median imputation and encoding categorical features.


In [5]:
# STEP 7: PREPROCESSOR PIPELINE
num_cols = ['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']
cat_cols = ['Sex', 'Embarked']

num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median'))])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])
print("✅ STEP 7: Preprocessor ready.")


✅ STEP 7: Preprocessor ready.


## ⚠️ STEP 8: BASELINE DUEL (DUMMY VS GRADIENT BOOSTING)
Demonstrating superiority over the majority class baseline.


In [6]:
# STEP 8: BASELINE DUEL
dummy_pipe = Pipeline([
    ('prep', preprocessor),
    ('dummy', DummyClassifier(strategy='most_frequent'))
])
dummy_pipe.fit(X_train, y_train)

gbc_base = Pipeline([
    ('prep', preprocessor),
    ('gbc', GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42))
])
gbc_base.fit(X_train, y_train)

acc_dummy = accuracy_score(y_test, dummy_pipe.predict(X_test))
acc_gbc = accuracy_score(y_test, gbc_base.predict(X_test))

print("="*65)
print(f"Baseline Majority Classifier Accuracy : {acc_dummy * 100:.2f}%")
print(f"Default GradientBoosting Accuracy     : {acc_gbc * 100:.2f}%")
print(f"Performance Advantage                 : +{(acc_gbc - acc_dummy) * 100:.2f}%")
print("="*65)


Baseline Majority Classifier Accuracy : 61.45%
Default GradientBoosting Accuracy     : 81.01%
Performance Advantage                 : +19.55%


## 🎯 STEP 9: 5-FOLD CV HYPERPARAMETER GRID TUNING
Optimizing `learning_rate`, `n_estimators`, and `max_depth`.


In [7]:
# STEP 9: GRID SEARCH CV
param_grid = {
    'gbc__n_estimators': [50, 100, 150],
    'gbc__learning_rate': [0.03, 0.05, 0.1],
    'gbc__max_depth': [2, 3, 4],
    'gbc__subsample': [0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(gbc_base, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold Cross-Validated ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'gbc__learning_rate': 0.05, 'gbc__max_depth': 3, 'gbc__n_estimators': 150, 'gbc__subsample': 0.8}
🎯 Best 5-Fold Cross-Validated ROC-AUC: 88.87%


## 📊 STEP 10 & 11: MULTI-METRIC EVALUATION & RESIDUAL DIAGNOSTICS
Comprehensive classification metrics and confusion matrix.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("📋 COMPREHENSIVE CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred, target_names=['Perished', 'Survived']))
print(f"Test Set ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print(f"Test Set Accuracy     : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"Test Set F1-Score     : {f1_score(y_test, y_pred):.4f}")


📋 COMPREHENSIVE CLASSIFICATION REPORT:
              precision    recall  f1-score   support

    Perished       0.80      0.90      0.85       110
    Survived       0.80      0.65      0.72        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.78       179
weighted avg       0.80      0.80      0.80       179

Test Set ROC-AUC Score: 0.8178
Test Set Accuracy     : 80.45%
Test Set F1-Score     : 0.7200


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & SMOKE TEST
Deploying the model to disk and validating sub-millisecond inference latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/titanic_gbc_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_passenger = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_class = loaded_pipe.predict(sample_passenger)[0]
pred_prob = loaded_pipe.predict_proba(sample_passenger)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🚢 LIVE PASSENGER PREDICTION:")
print(f"   Predicted Outcome: {'✅ SURVIVED' if pred_class == 1 else '❌ PERISHED'}")
print(f"   Survival Probability: {pred_prob*100:.2f}%")
print(f"   Inference Latency   : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/titanic_gbc_pipeline.joblib (60,346 bytes)

🚢 LIVE PASSENGER PREDICTION:
   Predicted Outcome: ❌ PERISHED
   Survival Probability: 9.13%
   Inference Latency   : 8.803 ms (SLA < 2.0ms: CHECK)
